# 指数权重清单推算

使用最近一次官方权重清单、成分股日收盘价和公司送转股行为，推算目标交易日的指数权重清单。

## 参数设置

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import gogoal_query
from utils.weight_projection import (
    fetch_corporate_actions,
    fetch_stock_closes_range,
    load_or_download_weight_source,
    make_projection_output_dir,
    normalize_date_key,
    project_weights_by_close_and_actions,
    save_projection_outputs,
    select_weight_source_for_target,
)

# 指数代码，例如 000016 / 000300 / 000905 / 000852。
INDEX_CODE = "000300"

# 目标日期，YYYYMMDD。
target_date = "20260722"

# 本地历史权重清单日期。选择规则为：日期 <= target_date，并取最接近 target_date 的日期。
# 新增历史清单时，在这里追加 YYYYMMDD；文件名须为 沪深300_样本权重_YYYYMMDD.csv。
HISTORICAL_WEIGHT_DATES = [
    "20250731", "20250829", "20250930", "20251031",
    "20251128", "20251231", "20260130", "20260227",
    "20260331", "20260430", "20260529", "20260630",
]
historical_weight_dir = PROJECT_ROOT / "data" / "weights_projection"
historical_weight_files = {
    date: historical_weight_dir / f"沪深300_样本权重_{date}.csv"
    for date in HISTORICAL_WEIGHT_DATES
}

# 输出目录；None 时自动写入 ./data/weights_projection/。
# 基准日和之后的每个交易日均保存为 f"{INDEX_CODE}-{current_date}.csv"。
output_dir = None

In [2]:
from xtquant import xtdata
xtdata.reconnect(port = 58610)

***** xtdata连接成功 2026-07-23 13:55:36*****
服务信息: {'tag': 'qmt_research', 'version': '1.0'}
服务地址: 127.0.0.1:58610
数据路径: E:\迅投极速交易终端睿智融科版\datadir
设置xtdata.enable_hello = False可隐藏此消息



## 读取或下载源权重清单

In [3]:
download_dir = PROJECT_ROOT / "data" / INDEX_CODE / "weight_projection" / "source_files"
latest_weights, latest_source_path = load_or_download_weight_source(
    index_code=INDEX_CODE,
    file_path=None,
    output_dir=download_dir,
)
target_date = normalize_date_key(target_date)

source_selection = select_weight_source_for_target(
    target_date=target_date,
    latest_weights=latest_weights,
    latest_source_path=latest_source_path,
    historical_weight_files=historical_weight_files,
)
source_weights = source_selection.weights
source_weight_path = source_selection.source_path
source_weight_date = source_selection.source_date

print("Latest downloaded weight date:", source_selection.latest_source_date)
print("Selection mode:", source_selection.selection_mode)
print("Source weight path:", source_weight_path)
print("Source weight date:", source_weight_date)
print("Target date:", target_date)
print("Source weight rows:", len(source_weights))
print("Source weight sum pct:", source_weights["raw_weight_pct"].sum())
display(source_weights.head())


正在下载指数 000300 样本权重...
解析到真实链接: https://oss-ch.csindex.com.cn/static/html/csindex/public/uploads/file/autofile/closeweight/000300closeweight.xls
下载完成: E:\Codex\系统\Stock-Index-Fitting\data\000300\weight_projection\source_files\000300_样本权重_20260723.xls
     日期Date  指数代码 Index Code 指数名称 Index Name 指数英文名称Index Name(Eng)  \
0  20260630              300           沪深300               CSI 300   
1  20260630              300           沪深300               CSI 300   
2  20260630              300           沪深300               CSI 300   
3  20260630              300           沪深300               CSI 300   
4  20260630              300           沪深300               CSI 300   

   成份券代码Constituent Code 成份券名称Constituent Name  \
0                      1                  平安银行   
1                      2                   万科A   
2                     63                  中兴通讯   
3                    100                 TCL科技   
4                    157                  中联重科   

                        成份券英

,stock_code,stock_name,raw_weight_pct,closeweight_data_date,source_path
0,000001.SZ,平安银行,0.345,20260630,E:\Codex\系统\Stock-Index-Fitting\data\000300\we...
1,000002.SZ,万科A,0.071,20260630,E:\Codex\系统\Stock-Index-Fitting\data\000300\we...
2,000063.SZ,中兴通讯,0.412,20260630,E:\Codex\系统\Stock-Index-Fitting\data\000300\we...
3,000100.SZ,TCL科技,0.428,20260630,E:\Codex\系统\Stock-Index-Fitting\data\000300\we...
4,000157.SZ,中联重科,0.122,20260630,E:\Codex\系统\Stock-Index-Fitting\data\000300\we...


## 查询日收盘价与公司行为

In [4]:
#Task5
def normalize_stock_code(value) -> str:
    raw = str(value).strip().upper()
    if raw.endswith((".SH", ".SZ", ".BJ")):
        return raw
    digits = raw.split(".")[0].zfill(6)
    return f"{digits}.SH" if digits.startswith(("5", "6", "9")) else f"{digits}.SZ"

def strip_market_suffix(value) -> str:
    return str(value).strip().upper().split(".")[0].zfill(6)

def normalize_trade_date_dash(value) -> str:
    return pd.Timestamp(value).strftime("%Y-%m-%d")

def sql_literal(value) -> str:
    return "'" + str(value).replace("'", "''") + "'"


def sql_in(values) -> str:
    vals = list(values)
    if not vals:
        raise ValueError("SQL IN 列表为空")
    return ", ".join(sql_literal(v) for v in vals)

def fetch_gogoal_corporate_actions(stock_codes: list[str], start_date: str, end_date: str) -> pd.DataFrame:
    stripped_codes = [strip_market_suffix(code) for code in stock_codes]
    sql = f"""
        SELECT
            stock_code,
            stock_name,
            declare_date,
            xr_xd_date AS ex_date,
            beftax_maxcashdiv,
            beftax_mincashdiv,
            aftax_cashdiv,
            stockdiv_ratio,
            trans_ratio,
            bonus_ratio,
            is_newest,
            is_valid
        FROM bas_stk_hisdistribution
        WHERE stock_code IN ({sql_in(stripped_codes)})
          AND xr_xd_date >= {sql_literal(normalize_trade_date_dash(start_date))}
          AND xr_xd_date <= {sql_literal(normalize_trade_date_dash(end_date))}
          AND is_valid = 1
    """
    raw = gogoal_query(sql, output_format="dataframe")
    if raw.empty:
        return pd.DataFrame(columns=["stock_code", "ex_date"])
    raw["stock_code"] = raw["stock_code"].map(normalize_stock_code)
    raw["ex_date"] = pd.to_datetime(raw["ex_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    return raw

stock_codes = source_weights["stock_code"].tolist()
# future_date = (pd.Timestamp(last_fitting_date) + timedelta(days=1)).strftime("%Y%m%d")
started_at = datetime.now().astimezone()
import_time = started_at.strftime("%Y%m%d-%H%M")

df_actions_gogoal = fetch_gogoal_corporate_actions(stock_codes, source_weight_date, target_date)
df_actions_gogoal["import_time"] = import_time

xt_action_frames = []
for stock_code in stock_codes:
    frame = xtdata.get_divid_factors(stock_code, source_weight_date, target_date)
    if frame is not None and not frame.empty:
        item = frame.copy().reset_index().rename(columns={"index": "ex_date"})
        item.insert(0, "stock_code", stock_code)
        xt_action_frames.append(item)
df_actions_xt = pd.concat(xt_action_frames, ignore_index=True) if xt_action_frames else pd.DataFrame(columns=["stock_code", "ex_date"])
if not df_actions_xt.empty:
    df_actions_xt["stock_code"] = df_actions_xt["stock_code"].map(normalize_stock_code)
    df_actions_xt["ex_date"] = pd.to_datetime(df_actions_xt["ex_date"], errors="coerce").dt.strftime("%Y-%m-%d")
df_actions_xt["import_time"] = import_time

left_keys = df_actions_gogoal[["stock_code", "ex_date"]].drop_duplicates()
right_keys = df_actions_xt[["stock_code", "ex_date"]].drop_duplicates()
df_actions_check = left_keys.merge(right_keys, on=["stock_code", "ex_date"], how="outer", indicator=True)
df_actions_check["is_consistent"] = df_actions_check["_merge"].eq("both")
df_actions_check["import_time"] = import_time
if df_actions_gogoal.empty and df_actions_xt.empty:
    df_actions_check = pd.DataFrame([{
        "stock_code": None,
        "ex_date": None,
        "_merge": "both",
        "is_consistent": True,
        "note": "Go-Goal 与迅投在查询窗口内均无公司行为事件",
        "import_time": import_time,
    }])

df_actions_merged = df_actions_gogoal.merge(
    df_actions_xt, on=["stock_code", "ex_date"], how="outer", suffixes=("_gogoal", "_xt")
)
df_actions_merged["import_time"] = import_time

daily_closes = fetch_stock_closes_range(
    gogoal_query,
    stock_codes=stock_codes,
    start_date=source_weight_date,
    end_date=target_date,
)

print("Daily close rows:", len(daily_closes))
print("Trading dates:", daily_closes["trade_date"].nunique())
print("Corporate action rows:", len(df_actions_merged))
display(daily_closes.head())
display(df_actions_merged.head())

正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


Daily close rows: 5100
Trading dates: 17
Corporate action rows: 82


,trade_date,stock_code,stock_name,close_price
0,20260630,300628.SZ,亿联网络,33.82
1,20260630,002466.SZ,天齐锂业,61.32
2,20260630,600039.SH,四川路桥,7.82
3,20260630,600219.SH,南山铝业,3.99
4,20260630,300896.SZ,爱美客,87.20


,stock_code,stock_name,declare_date,ex_date,beftax_maxcashdiv,beftax_mincashdiv,aftax_cashdiv,stockdiv_ratio,trans_ratio,bonus_ratio,...,time,interest,stockBonus,stockGift,allotNum,allotPrice,gugai,dr,import_time_xt,import_time
0,000157.SZ,中联重科,2026-07-14,2026-07-22,2.00000,None,2.00000,None,NaN,None,...,1.784650e+12,0.200000,0.0,0.0,0.0,0.0,0.0,1.027510,20260723-1336,20260723-1336
1,000425.SZ,徐工机械,2026-06-26,2026-07-03,2.00000,None,2.00000,None,NaN,None,...,1.783008e+12,0.200000,0.0,0.0,0.0,0.0,0.0,1.025220,20260723-1336,20260723-1336
2,000596.SZ,古井贡酒,2026-07-10,2026-07-17,34.00000,None,34.00000,None,NaN,None,...,1.784218e+12,3.400000,0.0,0.0,0.0,0.0,0.0,1.039938,20260723-1336,20260723-1336
3,000625.SZ,长安汽车,2026-07-10,2026-07-16,1.16823,None,1.16823,None,NaN,None,...,1.784131e+12,0.116823,0.0,0.0,0.0,0.0,0.0,1.017595,20260723-1336,20260723-1336
4,000657.SZ,中钨高新,2026-06-27,2026-07-06,2.30000,None,2.30000,None,NaN,None,...,1.783267e+12,0.230000,0.0,0.0,0.0,0.0,0.0,1.002376,20260723-1336,20260723-1336


In [5]:
# stock_codes = source_weights["stock_code"].tolist()

# daily_closes = fetch_stock_closes_range(
#     gogoal_query,
#     stock_codes=stock_codes,
#     start_date=source_weight_date,
#     end_date=target_date,
# )
# corporate_actions = fetch_corporate_actions(
#     gogoal_query,
#     stock_codes=stock_codes,
#     start_date=source_weight_date,
#     end_date=target_date,
# )

# print("Daily close rows:", len(daily_closes))
# print("Trading dates:", daily_closes["trade_date"].nunique())
# print("Corporate action rows:", len(corporate_actions))
# display(daily_closes.head())
# display(corporate_actions.head())


## 推算并保存权重清单

In [6]:
corporate_actions = df_actions_merged
target_weights, trace, daily_summary, standardized_actions = project_weights_by_close_and_actions(
    source_weights=source_weights,
    daily_closes=daily_closes,
    corporate_actions=corporate_actions,
    target_date=target_date,
)

if output_dir is None:
    output_dir = make_projection_output_dir(PROJECT_ROOT)

projection_outputs = save_projection_outputs(
    target_weights=target_weights,
    trace=trace,
    daily_summary=daily_summary,
    source_weights=source_weights,
    standardized_actions=standardized_actions,
    output_dir=output_dir,
    target_date=target_date,
    index_code=INDEX_CODE,
)

print("Output dir:", projection_outputs.output_dir)
print("Target weights:", projection_outputs.target_path)
print("Daily summary:", projection_outputs.summary_path)
print("Saved projected weight files:", len(trace))
print("Projected weight sum pct:", target_weights["projected_weight_pct"].sum())
display(daily_summary.tail())
display(target_weights.head(20))


Output dir: E:\Codex\系统\Stock-Index-Fitting\data\weights_projection
Target weights: E:\Codex\系统\Stock-Index-Fitting\data\weights_projection\000300-20260722.csv
Daily summary: E:\Codex\系统\Stock-Index-Fitting\data\weights_projection\000300-20260722-daily_summary.csv
Saved projected weight files: 17
Projected weight sum pct: 100.0


,trade_date,stock_count,weight_sum_pct,action_count,share_action_count,cash_dividend_action_count
12,20260716,300,100.0,8,0,8
13,20260717,300,100.0,9,0,9
14,20260720,300,100.0,5,0,5
15,20260721,300,100.0,2,0,2
16,20260722,300,100.0,2,0,2


,trade_date,stock_code,stock_name,source_weight_date,source_weight_pct,projected_weight_pct,base_close,close_price,share_factor,relative_value,cumulative_cash_dividend_per_base_share
0,20260722,300308.SZ,中际旭创,20260630,5.008,4.413640,1270.00,1060.80,1.0,0.041831,0.000000
1,20260722,300750.SZ,宁德时代,20260630,3.675,3.672848,393.01,372.26,1.0,0.034810,0.000000
2,20260722,600519.SH,贵州茅台,20260630,2.620,3.043104,1185.49,1305.00,1.0,0.028841,0.000000
3,20260722,300502.SZ,新易盛,20260630,2.992,2.646201,607.00,508.80,1.0,0.025080,0.000000
4,20260722,601318.SH,中国平安,20260630,1.799,2.145078,47.74,53.95,1.0,0.020330,0.000000
5,20260722,601899.SH,紫金矿业,20260630,1.465,1.979844,25.14,32.20,1.0,0.018764,0.000000
6,20260722,600036.SH,招商银行,20260630,1.554,1.787922,35.50,38.71,1.0,0.016945,1.002999
7,20260722,000333.SZ,美的集团,20260630,1.299,1.538822,75.53,84.80,1.0,0.014584,0.000000
8,20260722,688256.SH,寒武纪,20260630,1.772,1.535067,1595.55,1310.00,1.0,0.014549,0.000000
9,20260722,600900.SH,长江电力,20260630,1.145,1.327694,26.47,29.09,1.0,0.012583,0.790000


## 可选：查看某一天的推算轨迹

In [7]:
# 已保存源权重日、其后至 target_date 当日的全部交易日推算结果。
# 每个文件表示：使用 current_date 闭市后的 close price 估计次日指数将使用的权重清单。
if trace:
    saved_dates = sorted(trace)
    print("Saved date range:", saved_dates[0], "->", saved_dates[-1])
    print("Saved files:", len(saved_dates))
    display(trace[saved_dates[-1]].head(20))
else:
    print("没有中间交易日需要保存。")


Saved date range: 20260630 -> 20260722
Saved files: 17


,trade_date,stock_code,stock_name,source_weight_date,source_weight_pct,projected_weight_pct,base_close,close_price,share_factor,relative_value,cumulative_cash_dividend_per_base_share
0,20260722,300308.SZ,中际旭创,20260630,5.008,4.413640,1270.00,1060.80,1.0,0.041831,0.000000
1,20260722,300750.SZ,宁德时代,20260630,3.675,3.672848,393.01,372.26,1.0,0.034810,0.000000
2,20260722,600519.SH,贵州茅台,20260630,2.620,3.043104,1185.49,1305.00,1.0,0.028841,0.000000
3,20260722,300502.SZ,新易盛,20260630,2.992,2.646201,607.00,508.80,1.0,0.025080,0.000000
4,20260722,601318.SH,中国平安,20260630,1.799,2.145078,47.74,53.95,1.0,0.020330,0.000000
5,20260722,601899.SH,紫金矿业,20260630,1.465,1.979844,25.14,32.20,1.0,0.018764,0.000000
6,20260722,600036.SH,招商银行,20260630,1.554,1.787922,35.50,38.71,1.0,0.016945,1.002999
7,20260722,000333.SZ,美的集团,20260630,1.299,1.538822,75.53,84.80,1.0,0.014584,0.000000
8,20260722,688256.SH,寒武纪,20260630,1.772,1.535067,1595.55,1310.00,1.0,0.014549,0.000000
9,20260722,600900.SH,长江电力,20260630,1.145,1.327694,26.47,29.09,1.0,0.012583,0.790000
